# BirdCLEF 2026 - PyTorch CNN Pipeline (No Leakage, Multi-label, CNN)
Based on PLAN.md recommendations. Modified for OFFLINE Kaggle use.


In [1]:
import os, gc, random, warnings, time
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
import timm

warnings.filterwarnings('ignore')
print('Libraries loaded')



Libraries loaded


In [2]:
# ── Config ─────────────────────────────────────────────────────────────
BASE = Path('/kaggle/input/competitions/birdclef-2026')
TRAIN_AUDIO = BASE / 'train_audio'
TRAIN_SND = BASE / 'train_soundscapes'
TEST_SND = BASE / 'test_soundscapes'
OUT = Path('/kaggle/working')

# UPDATE THIS PATH to point to your attached Kaggle dataset containing the .pth file
# (e.g. '/kaggle/input/efficientnet-dataset/tf_efficientnet_b2_aa-60c94f97.pth')
PRETRAINED_WEIGHTS_PATH = '/kaggle/input/REPLACE_WITH_DATASET_NAME/tf_efficientnet_b2_aa-60c94f97.pth'

SR = 32_000
SEGMENT_SEC = 5
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 512
FMIN = 50
FMAX = 14_000

MAX_CLIPS_PER_SPECIES = 30
MIN_RATING = 3.0
N_FOLDS = 5
SEED = 42

BATCH_SIZE = 32
EPOCHS = 5
LR = 1e-3
WEIGHT_DECAY = 1e-4

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')



Using device: cpu


## 1. Load Data & Taxonomy


In [3]:
train_df = pd.read_csv(BASE / 'train.csv')
taxonomy = pd.read_csv(BASE / 'taxonomy.csv')
snd_labels = pd.read_csv(BASE / 'train_soundscapes_labels.csv')
sample_sub = pd.read_csv(BASE / 'sample_submission.csv')

SPECIES = [c for c in sample_sub.columns if c != 'row_id']
n_classes = len(SPECIES)
species_to_idx = {sp: i for i, sp in enumerate(SPECIES)}
idx_to_species = {i: sp for i, sp in enumerate(SPECIES)}

print(f'Classes: {n_classes}')



Classes: 234


## 2. Prepare Data (No Leakage, True Multi-Label, Grouping)


In [4]:
# Filter train_audio
target_set = set(SPECIES)
train_df = train_df[train_df['primary_label'].isin(target_set)].copy()
train_df = train_df[(train_df['rating'] == 0) | (train_df['rating'] >= MIN_RATING)].copy()

if MAX_CLIPS_PER_SPECIES:
    train_df = (
        train_df.groupby('primary_label', group_keys=False)
        .apply(lambda g: g.sample(min(len(g), MAX_CLIPS_PER_SPECIES), random_state=SEED))
    ).reset_index(drop=True)

# Create targets for train_audio (single label becomes one-hot in multi-label context)
train_df['filepath'] = train_df['filename'].apply(lambda x: str(TRAIN_AUDIO / x))
train_df['group'] = train_df['filename'] # Grouping by filename
train_df['start_sec'] = 0.0 # Will dynamically slice in dataset
train_df['is_soundscape'] = False

# Soundscapes
def seconds_from_hms(hms_str):
    h, m, s = map(int, str(hms_str).split(':'))
    return h*3600 + m*60 + s

snd_labels['start_sec'] = snd_labels['start'].apply(seconds_from_hms)
snd_labels['filepath'] = snd_labels['filename'].apply(lambda x: str(TRAIN_SND / x))
snd_labels['group'] = snd_labels['filename']
snd_labels['is_soundscape'] = True

# Convert semicolon-separated primary_label to multi-label
snd_labels['labels_list'] = snd_labels['primary_label'].apply(lambda x: [sp.strip() for sp in str(x).split(';') if sp.strip() in target_set])

snd_labels['stratify_label'] = snd_labels['labels_list'].apply(lambda x: x[0] if len(x) > 0 else 'nocall')

train_df['labels_list'] = train_df['primary_label'].apply(lambda x: [x])
train_df['stratify_label'] = train_df['primary_label']

cols = ['filepath', 'start_sec', 'labels_list', 'stratify_label', 'group', 'is_soundscape']
full_df = pd.concat([train_df[cols], snd_labels[cols]], ignore_index=True)

# Build multi-label target matrix
targets = np.zeros((len(full_df), n_classes), dtype=np.float32)
for i, labels in enumerate(full_df['labels_list']):
    for sp in labels:
        targets[i, species_to_idx[sp]] = 1.0
        
full_df['target'] = list(targets)

# KFold
skf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
full_df['fold'] = -1
for fold, (tr_idx, val_idx) in enumerate(skf.split(full_df, full_df['stratify_label'], groups=full_df['group'])):
    full_df.loc[val_idx, 'fold'] = fold

print("Folds distribution:")
print(full_df['fold'].value_counts())



Folds distribution:
fold
0    1402
1    1362
3    1356
2    1329
4    1319
Name: count, dtype: int64


## 3. Dataset & Augmentations


In [5]:
class BirdDataset(Dataset):
    def __init__(self, df, is_train=True):
        self.df = df
        self.is_train = is_train
        
        self.mel_spec = T.MelSpectrogram(
            sample_rate=SR,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS,
            f_min=FMIN,
            f_max=FMAX
        )
        self.amplitude_to_db = T.AmplitudeToDB()
        
        # Augmentations (Time & Freq Masking)
        self.time_mask = T.TimeMasking(time_mask_param=30)
        self.freq_mask = T.FrequencyMasking(freq_mask_param=15)
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filepath = row['filepath']
        is_snd = row['is_soundscape']
        
        try:
            if is_snd:
                start_frame = int(row['start_sec'] * SR)
                frames_to_read = int(SEGMENT_SEC * SR)
                y, sr_orig = torchaudio.load(filepath, frame_offset=start_frame, num_frames=frames_to_read)
            else:
                y, sr_orig = torchaudio.load(filepath)
                if y.shape[1] > SR * SEGMENT_SEC:
                    if self.is_train:
                        start = random.randint(0, y.shape[1] - SR * SEGMENT_SEC)
                    else:
                        start = 0
                    y = y[:, start:start + SR * SEGMENT_SEC]
                
            if sr_orig != SR:
                y = torchaudio.functional.resample(y, orig_freq=sr_orig, new_freq=SR)
                
            if y.shape[0] > 1:
                y = y.mean(dim=0, keepdim=True)
                
            target_length = SR * SEGMENT_SEC
            if y.shape[1] < target_length:
                y = F.pad(y, (0, target_length - y.shape[1]))
                
        except Exception as e:
            y = torch.zeros(1, SR * SEGMENT_SEC)
            
        mel = self.mel_spec(y)
        mel = self.amplitude_to_db(mel)
        
        if self.is_train:
            mel = self.time_mask(mel)
            mel = self.freq_mask(mel)
            
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        mel = mel.repeat(3, 1, 1)
        
        target = torch.tensor(row['target'], dtype=torch.float32)
        return mel, target



## 4. Model Definition (EfficientNet-B2 for Offline Usage)


In [6]:
class BirdModel(nn.Module):
    def __init__(self, num_classes=n_classes, model_name='tf_efficientnet_b2', pretrained=False):
        super().__init__()
        # Initialize without downloading from huggingface
        self.backbone = timm.create_model(model_name, pretrained=pretrained, in_chans=3)
        in_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Identity() # Remove original classifier
        
        self.head = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(in_features, num_classes)
        )
        
    def forward(self, x):
        features = self.backbone(x)
        output = self.head(features)
        return output

def build_model():
    model = BirdModel(pretrained=False).to(device)
    # Attempt to load the offline backbone weights
    if os.path.exists(PRETRAINED_WEIGHTS_PATH):
        print(f"Loading pretrained backbone from {PRETRAINED_WEIGHTS_PATH}")
        try:
            model.backbone.load_state_dict(torch.load(PRETRAINED_WEIGHTS_PATH, map_location=device), strict=False)
        except Exception as e:
            print(f"Failed to load weights: {e}")
    else:
        print(f"WARNING: Offline weights not found at {PRETRAINED_WEIGHTS_PATH}")
        print("Please update the PRETRAINED_WEIGHTS_PATH in the Config cell to point to your dataset.")
    return model



## 5. Training Loop & Validation


In [7]:
def train_fold(fold):
    print(f"\n{'='*20} Fold {fold} {'='*20}")
    train_df_fold = full_df[full_df['fold'] != fold].reset_index(drop=True)
    val_df_fold = full_df[full_df['fold'] == fold].reset_index(drop=True)
    
    train_dataset = BirdDataset(train_df_fold, is_train=True)
    val_dataset = BirdDataset(val_df_fold, is_train=False)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    model = build_model()
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    best_auc = 0
    oof_preds = []
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1} Train", leave=False):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            
            out = model(x)
            loss = criterion(out, y)
            
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        scheduler.step()
        
        model.eval()
        val_loss = 0
        val_preds = []
        val_targets = []
        with torch.no_grad():
            for x, y in tqdm(val_loader, desc=f"Epoch {epoch+1} Val", leave=False):
                x, y = x.to(device), y.to(device)
                out = model(x)
                loss = criterion(out, y)
                val_loss += loss.item()
                val_preds.append(torch.sigmoid(out).cpu().numpy())
                val_targets.append(y.cpu().numpy())
                
        val_preds = np.vstack(val_preds)
        val_targets = np.vstack(val_targets)
        
        aucs = []
        for i in range(n_classes):
            if val_targets[:, i].sum() > 0:
                try:
                    auc = roc_auc_score(val_targets[:, i], val_preds[:, i])
                    aucs.append(auc)
                except ValueError:
                    pass
        val_auc = np.mean(aucs) if len(aucs) > 0 else 0
        
        print(f"Epoch {epoch+1}: Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f} | Val AUC: {val_auc:.4f}")
        
        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), OUT / f"fold_{fold}_best.pth")
            
    print(f"Best Fold {fold} AUC: {best_auc:.4f}")
    
    model.load_state_dict(torch.load(OUT / f"fold_{fold}_best.pth"))
    model.eval()
    val_preds = []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            out = model(x)
            val_preds.append(torch.sigmoid(out).cpu().numpy())
    val_preds = np.vstack(val_preds)
    
    # Save model explicitly to /kaggle/working as requested
    torch.save(model.state_dict(), f'/kaggle/working/birdclef_model_fold_{fold}.pth')
    return val_preds, val_df_fold

# Train all folds
oof_dfs = []
for fold in range(N_FOLDS):
    # Only training fold 0 for speed in demonstration/baseline.
    # To fully utilize, run all folds (remove the break).
    val_preds, val_df_fold = train_fold(fold)
    val_df_fold[[f'pred_{i}' for i in range(n_classes)]] = val_preds
    oof_dfs.append(val_df_fold)
    break # Remove this break to train all folds




==================== Fold 0 ====================
Please update the PRETRAINED_WEIGHTS_PATH in the Config cell to point to your dataset.


Epoch 1 Train:   0%|          | 0/167 [00:00<?, ?it/s]

Epoch 1 Val:   0%|          | 0/44 [00:00<?, ?it/s]

Epoch 1: Train Loss: 0.0533 | Val Loss: 0.0349 | Val AUC: 0.6297


Epoch 2 Train:   0%|          | 0/167 [00:00<?, ?it/s]

...

Epoch 4 Val:   0%|          | 0/44 [00:00<?, ?it/s]

Epoch 4: Train Loss: 0.0256 | Val Loss: 0.0339 | Val AUC: 0.7283


Epoch 5 Train:   0%|          | 0/167 [00:00<?, ?it/s]

Epoch 5 Val:   0%|          | 0/44 [00:00<?, ?it/s]

Epoch 5: Train Loss: 0.0240 | Val Loss: 0.0327 | Val AUC: 0.7573
Best Fold 0 AUC: 0.7573


## 6. Threshold Optimization (Optional, disabled for now to just output probabilities)


In [8]:
# You can optimize class thresholds using oof_dfs
# For AUC, probabilities are sufficient. Thresholds matter for F1 or macro F1.
# Since eval metric is ROC-AUC, we leave probabilities as they are.



## 7. Inference & Submission (Fixing the Test Fallback Bug)


In [9]:
import concurrent.futures
import soundfile as sf
import math

test_files = sorted(TEST_SND.glob('*.ogg'))
IS_DRY_RUN = len(test_files) == 0

if IS_DRY_RUN:
    print("No hidden test files found. Dry-run on 10 train soundscapes...")
    test_files = sorted(TRAIN_SND.glob('*.ogg'))[:10]
else:
    print(f'Hidden test soundscapes found: {len(test_files)}')

model = BirdModel(pretrained=False).to(device)
try:
    model.load_state_dict(torch.load(OUT / "fold_0_best.pth", map_location=device))
    print("Loaded trained fold 0 successfully.")
except Exception as e:
    print(f"WARNING: Could not load trained fold model. {e}")
    
model.eval()

mel_spec = T.MelSpectrogram(
    sample_rate=SR,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    f_min=FMIN,
    f_max=FMAX
).to(device)
amplitude_to_db = T.AmplitudeToDB().to(device)

def read_audio(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    # Ensure it's not empty and right sample rate is assumed (Kaggle tests are 32k)
    return y

BATCH_FILES = 8 # Process 8 files (8 * 12 = 96 windows) at a time
N_WINDOWS_PER_FILE = 12

all_rows = []
probs_list = []

# Using ThreadPoolExecutor for background file reading
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as io_exec:
    for i in tqdm(range(0, len(test_files), BATCH_FILES), desc="Inferring test batches"):
        batch_paths = test_files[i:i+BATCH_FILES]
        
        # Multithreaded read
        future_audio = [io_exec.submit(read_audio, p) for p in batch_paths]
        batch_audio = [f.result() for f in future_audio]
        
        batch_mels = []
        batch_row_ids = []
        
        for bi, path in enumerate(batch_paths):
            fname = path.stem
            y = batch_audio[bi]
            
            y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(0).to(device)
            duration = y_tensor.shape[1] / SR
            n_segments = max(1, int(np.ceil(duration / SEGMENT_SEC)))
            
            for seg_i in range(n_segments):
                start = seg_i * SEGMENT_SEC
                end = start + SEGMENT_SEC
                end_time = int(end)
                row_id = f'{fname}_{end_time}'
                
                start_frame = int(start * SR)
                end_frame = int(end * SR)
                segment = y_tensor[:, start_frame:end_frame]
                
                if segment.shape[1] < SR * SEGMENT_SEC:
                    segment = F.pad(segment, (0, SR * SEGMENT_SEC - segment.shape[1]))
                    
                mel = mel_spec(segment)
                mel = amplitude_to_db(mel)
                mel = (mel - mel.mean()) / (mel.std() + 1e-6)
                mel = mel.repeat(3, 1, 1) # Shape: (3, MELS, TIME)
                
                batch_mels.append(mel)
                batch_row_ids.append(row_id)
                
        if not batch_mels:
            continue
            
        # Batch inference
        batch_mels_tensor = torch.stack(batch_mels).to(device) # Shape: (B, 3, MELS, TIME)
        
        with torch.no_grad():
            # If batch is too large, you could split it here, but 8*12=96 is fine for B2
            out = model(batch_mels_tensor)
            batch_probs = torch.sigmoid(out).cpu().numpy()
            
        all_rows.extend(batch_row_ids)
        probs_list.append(batch_probs)

if len(all_rows) == 0:
    print("No predictions generated. Using baseline probabilities.")
    probs = np.empty((0, n_classes))
    row_ids = []
else:
    probs = np.vstack(probs_list)
    row_ids = all_rows

sub = pd.DataFrame(probs, columns=SPECIES)
sub.insert(0, 'row_id', row_ids)

# We must output exactly the rows in sample_submission, and handle dry_run gracefully
if IS_DRY_RUN:
    print("Dry-run: formatting submission to match sample_submission...")
    sample_pub = pd.read_csv(BASE / 'sample_submission.csv')
    mean_pred = sub[SPECIES].mean(axis=0).fillna(1.0/n_classes).to_dict()
    sub = sample_pub.copy()
    for sp in SPECIES:
        sub[sp] = mean_pred[sp]
else:
    sub = sample_sub[['row_id']].merge(sub, on='row_id', how='left')
    baseline_prob = 1.0 / n_classes
    sub[SPECIES] = sub[SPECIES].fillna(baseline_prob)

submission_path = OUT / 'submission.csv'
sub.to_csv(submission_path, index=False)
print(f'submission.csv saved -> shape {sub.shape}')


No hidden test files found. Dry-run on 10 train soundscapes...
Loaded trained fold 0 successfully.


Inferring test batches:   0%|          | 0/2 [00:00<?, ?it/s]

Dry-run: formatting submission to match sample_submission...
submission.csv saved -> shape (3, 235)
